In [1]:
import polars as pl
import json
from pathlib import Path

In [2]:
%cd ..

/data/projects/tcrio/tcrio


In [ ]:
data_path = Path("../reference/library-imgt/imgt.202614-2.sv8-1-gcaf67b3.mouse.inlined.json")
alleles = json.loads(data_path.read_text())
# data releseas: https://www.imgt.org/IMGT_vquest/data_releases
# Release 202614-2 (31 March 2026)

In [39]:
alleles = json.loads(data_path.read_text())

In [40]:
pl.DataFrame(alleles)["geneType"].value_counts()

geneType,count
str,u32
"""V""",1241
"""J""",102
"""C""",58
"""D""",66


In [ ]:
from dataclasses import dataclass
from typing import Literal


@dataclass
class TcrAllele:
    species: Literal["human", "mouse"]
    gene_type: Literal["V", "D", "J"]
    name: str
    gene : str
    chain : str
    sequence_nt: str
    cdr3_part_nt: str
    is_functional: bool

allele_list = []

for species, data_path in zip(["human", "mouse"], [
    Path("../reference/library-imgt/imgt.202614-2.sv8-1-gcaf67b3.human.inlined.json"),
    Path("../reference/library-imgt/imgt.202614-2.sv8-1-gcaf67b3.mouse.inlined.json")
]):
    alleles = json.loads(data_path.read_text())
    for allele in alleles:
        try:
            if allele["geneType"] == "C":
                continue

            if allele["chains"] in [["IGK"], ["IGL"], ["IGH"]]:
                continue
            
            for chain in allele["chains"]:
                if allele["geneType"] == "V":
                    start, end = allele["anchorPoints"]["CDR3Begin"], allele["anchorPoints"]["VEnd"]
                    cdr3_part_nt = allele["baseSequence"][start:end]
                elif allele["geneType"] == "J":
                    start, end = allele["anchorPoints"]["JBegin"], allele["anchorPoints"]["FR4Begin"]
                    cdr3_part_nt = allele["baseSequence"][start:end]
                if allele["geneType"] == "D":
                    cdr3_part_nt = allele["baseSequence"]

                name = allele["name"]
                gene = name.split("*")[0]

                a = TcrAllele(
                    species=species,
                    chain=chain,
                    gene_type=allele["geneType"],
                    name=name,
                    gene=gene,
                    sequence_nt=allele["baseSequence"],
                    cdr3_part_nt=cdr3_part_nt
                )
                allele_list.append(a)

        
        except KeyError:
            print(allele)

pl.DataFrame(allele_list).write_csv("../reference/library-imgt/imgt.202614-2.parsed.tsv", separator="\t")


{'baseSequence': 'GGACAAAACATTGACCAGCCCACTGAGATGACAGCTACGGAAGGTGCCATTGTCCAGATCAACTGCACGTACCAGACATCTGGGTTCAACGGGCTGTTCTGGTACCAGCAACATGCTGGCGAAGCACCCACATTTCTGTCTTACAATGTTCTGGATGGTCTGGAGGAGAAAGGTCG', 'name': 'TRAV1-2*02', 'geneType': 'V', 'isFunctional': True, 'chains': ['TRA'], 'meta': {'comments': "U32544|TRAV1-2*02|Homo sapiens|[F]|V-REGION|302..477|176 nt|1| | | | |176+48=224|partial in 3'| |"}, 'anchorPoints': {'FR1Begin': 0, 'CDR1Begin': 75, 'FR2Begin': 93, 'CDR2Begin': 144, 'FR3Begin': 162, 'VEnd': 176}}
{'baseSequence': 'AGCCAAAAGATAGAACAGAATTCCGAGGCCCTGAACATTCAGGAGGGTAAAACGGCCACCCTGACCTGCAACTATACAAACTATTCTCCAGCATACTTACAGTGGTACCGACAAGATCCAGGAAGAGGCCCTGTTTTCTTGCTACTCATACGTGAAAATGAGAAAGAAAAAAGGAAAGAAAGACTGAAGGTCACCTTTGATACCACCCTTAACCAGA', 'name': 'TRAV6*06', 'geneType': 'V', 'isFunctional': True, 'chains': ['TRA'], 'meta': {'comments': "U32542|TRAV6*06|Homo sapiens|[F]|V-REGION|174..390|217 nt|1| | | | |217+42=259|partial in 3'| |"}, 'anchorPoints': {'FR1Begin': 0, 'CDR1Begin': 78, 

In [33]:
[a for a in alleles if a["geneType"] == "V"]

[{'baseSequence': 'CAGGTGCAGCTGGTGCAGTCTGGGGCTGAGGTGAAGAAGCCTGGGGCCTCAGTGAAGGTCTCCTGCAAGGCTTCTGGATACACCTTCACCGGCTACTATATGCACTGGGTGCGACAGGCCCCTGGACAAGGGCTTGAGTGGATGGGACGGATCAACCCTAACAGTGGTGGCACAAACTATGCACAGAAGTTTCAGGGCAGGGTCACCAGTACCAGGGACACGTCCATCAGCACAGCCTACATGGAGCTGAGCAGGCTGAGATCTGACGACACGGTCGTGTATTACTGTGCGAGAGA',
  'name': 'IGHV1-2*01',
  'geneType': 'V',
  'isFunctional': True,
  'chains': ['IGH'],
  'meta': {'comments': 'X07448|IGHV1-2*01|Homo sapiens|F|V-REGION|269..564|296 nt|1| | | | |296+24=320| | |'},
  'anchorPoints': {'FR1Begin': 0,
   'CDR1Begin': 75,
   'FR2Begin': 99,
   'CDR2Begin': 150,
   'FR3Begin': 174,
   'CDR3Begin': 285,
   'VEnd': 296}},
 {'baseSequence': 'CAGGTGCAGCTGGTGCAGTCTGGGGCTGAGGTGAAGAAGCCTGGGGCCTCAGTGAAGGTCTCCTGCAAGGCTTCTGGATACACCTTCACCGGCTACTATATGCACTGGGTGCGACAGGCCCCTGGACAAGGGCTTGAGTGGATGGGATGGATCAACCCTAACAGTGGTGGCACAAACTATGCACAGAAGTTTCAGGGCAGGGTCACCATGACCAGGGACACGTCCATCAGCACAGCCTACATGGAGCTGAGCAGGCTGAGATCTGACGACACGGCCGTGTATTACTGTGCGAGAGA',
  'name': 'I